# Twitter Sentiment Analysis Using RNN

Week 31 Graded Mini Project  
Advanced Certificate Programme in Applied Artificial Intelligence and Machine Learning

## Objective

Build an end-to-end sentiment analysis pipeline for Twitter posts using a Recurrent Neural Network (RNN). The final model should classify tweets into one of three sentiment classes: `Negative`, `Neutral`, or `Positive`.

## Assignment And Rubric Checklist

This notebook is organized to match the project task list and rubric:

1. Load the dataset.
2. Clean and preprocess tweet text.
3. Engineer numerical text features and token sequences.
4. Conduct exploratory data analysis.
5. Build an embedding-based LSTM or GRU model.
6. Train, validate, and test the model.
7. Evaluate accuracy, precision, recall, and F1-score.
8. Plot learning curves and confusion matrix.
9. Improve the model through controlled hyperparameter experiments.
10. Present final findings and sample predictions.

## Dataset Notes

The source CSV file has no header row. It will be loaded with these explicit column names:

- `tweet_id`
- `entity`
- `sentiment`
- `tweet_text`

Initial profiling found four sentiment labels in the raw dataset: `Negative`, `Positive`, `Neutral`, and `Irrelevant`. Because the assignment objective and rubric describe a three-class sentiment task, the main modeling pipeline will document and exclude `Irrelevant`, then train on `Negative`, `Neutral`, and `Positive`.

## Phase 1: Project Setup

Status: scaffold created.

The implementation will use a reproducible project structure:

- `requirements.txt` for dependencies.
- `PROJECT_CONTEXT.md` for assignment, rubric, and dataset decisions.
- `outputs/figures` for generated charts.
- `outputs/models` for trained model files.
- `outputs/tables` for evaluation tables and intermediate summaries.
- `outputs/reports` for report or presentation-ready exports.

In [ ]:
# Phase 1 setup cell. Execute this before later phases.
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "twitter_training.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"
REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [FIGURES_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path exists: {DATA_PATH.exists()} -> {DATA_PATH}")

---

# Part 1: Data Processing

## Phase 2: Loading And Raw Data Validation

Phase 2 loads the raw CSV exactly as supplied and validates its structure before any cleaning is performed. This gives the project an auditable baseline for the rubric item `Loading the Dataset` and prepares data quality evidence for the next phase.

Work completed in this phase:

- Load `twitter_training.csv` into a pandas DataFrame with explicit column names.
- Confirm shape, columns, data types, and sample rows.
- Check missing values, blank tweet text, duplicate rows, and sentiment labels.
- Compute raw sentiment distribution and text length statistics.
- Save validation summaries to `outputs/tables` for documentation and later reporting.

No cleaning is applied in this phase. Cleaning starts in Phase 3.

In [ ]:
# Phase 2 imports and constants.
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 20)

RAW_COLUMNS = ["tweet_id", "entity", "sentiment", "tweet_text"]
ASSIGNMENT_SENTIMENTS = {"Negative", "Neutral", "Positive"}

print("Phase 2 constants ready.")
print(f"Expected raw columns: {RAW_COLUMNS}")

In [ ]:
# Load the no-header CSV using explicit column names.
df_raw = pd.read_csv(
    DATA_PATH,
    header=None,
    names=RAW_COLUMNS,
    encoding="utf-8",
)

print(f"Dataset loaded successfully: {DATA_PATH.name}")
print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
print("Columns:", list(df_raw.columns))

display(df_raw.head())

In [ ]:
# Structural validation: schema, data types, missing values, and unique counts.
schema_is_valid = list(df_raw.columns) == RAW_COLUMNS

dataset_overview = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "schema_matches_expected_columns",
            "unique_tweet_ids",
            "unique_entities",
            "unique_sentiments",
            "missing_tweet_text",
            "exact_duplicate_rows",
        ],
        "value": [
            len(df_raw),
            df_raw.shape[1],
            schema_is_valid,
            df_raw["tweet_id"].nunique(dropna=True),
            df_raw["entity"].nunique(dropna=True),
            df_raw["sentiment"].nunique(dropna=True),
            df_raw["tweet_text"].isna().sum(),
            df_raw.duplicated().sum(),
        ],
    }
)

missing_summary = pd.DataFrame(
    {
        "missing_count": df_raw.isna().sum(),
        "missing_percent": (df_raw.isna().mean() * 100).round(2),
        "unique_values": df_raw.nunique(dropna=True),
        "dtype": df_raw.dtypes.astype(str),
    }
).reset_index(names="column")

display(dataset_overview)
display(missing_summary)

In [ ]:
# Sentiment validation.
sentiment_distribution = (
    df_raw["sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="count")
)
sentiment_distribution["percent"] = (
    sentiment_distribution["count"] / len(df_raw) * 100
).round(2)

observed_sentiments = set(df_raw["sentiment"].dropna().unique())
extra_sentiments = sorted(observed_sentiments - ASSIGNMENT_SENTIMENTS)
missing_assignment_sentiments = sorted(ASSIGNMENT_SENTIMENTS - observed_sentiments)

sentiment_validation = pd.DataFrame(
    {
        "check": [
            "assignment_sentiments_present",
            "extra_labels_in_raw_data",
            "recommended_main_model_scope",
        ],
        "result": [
            len(missing_assignment_sentiments) == 0,
            ", ".join(extra_sentiments) if extra_sentiments else "None",
            "Use Negative, Neutral, and Positive; document and exclude Irrelevant before modeling.",
        ],
    }
)

display(sentiment_distribution)
display(sentiment_validation)

In [ ]:
# Duplicate and blank-text validation.
text_as_string = df_raw["tweet_text"].fillna("").astype(str)
blank_or_missing_text = text_as_string.str.strip().eq("")

duplicate_summary = pd.DataFrame(
    {
        "metric": [
            "blank_or_missing_tweet_text_rows",
            "exact_duplicate_rows",
            "duplicate_tweet_text_rows_non_null",
            "duplicate_tweet_text_and_sentiment_rows_non_null",
        ],
        "value": [
            int(blank_or_missing_text.sum()),
            int(df_raw.duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), "tweet_text"].duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), ["tweet_text", "sentiment"]].duplicated().sum()),
        ],
    }
)

blank_text_by_sentiment = (
    df_raw.loc[blank_or_missing_text, "sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="blank_or_missing_rows")
)

display(duplicate_summary)
display(blank_text_by_sentiment)

In [ ]:
# Raw tweet length statistics. These help set later sequence length choices.
df_raw_profile = df_raw.copy()
df_raw_profile["tweet_text_str"] = text_as_string
df_raw_profile["char_len"] = df_raw_profile["tweet_text_str"].str.len()
df_raw_profile["word_len"] = df_raw_profile["tweet_text_str"].str.split().str.len()

length_summary_overall = (
    df_raw_profile[["char_len", "word_len"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

length_summary_by_sentiment = (
    df_raw_profile
    .groupby("sentiment")[["char_len", "word_len"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
length_summary_by_sentiment.columns = [
    f"{feature}_{stat}" for feature, stat in length_summary_by_sentiment.columns.to_flat_index()
]

top_entities_raw = (
    df_raw["entity"]
    .value_counts()
    .rename_axis("entity")
    .reset_index(name="count")
)
top_entities_raw["percent"] = (top_entities_raw["count"] / len(df_raw) * 100).round(2)

display(length_summary_overall)
display(length_summary_by_sentiment)
display(top_entities_raw.head(15))

In [ ]:
# Save Phase 2 audit tables for the report and later phases.
phase2_tables = {
    "phase2_dataset_overview.csv": dataset_overview,
    "phase2_missing_summary.csv": missing_summary,
    "phase2_sentiment_distribution.csv": sentiment_distribution,
    "phase2_sentiment_validation.csv": sentiment_validation,
    "phase2_duplicate_summary.csv": duplicate_summary,
    "phase2_blank_text_by_sentiment.csv": blank_text_by_sentiment,
    "phase2_length_summary_overall.csv": length_summary_overall.reset_index(names="statistic"),
    "phase2_length_summary_by_sentiment.csv": length_summary_by_sentiment.reset_index(),
    "phase2_top_entities_raw.csv": top_entities_raw,
}

for filename, table in phase2_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

phase2_summary_lines = [
    "# Phase 2 Validation Summary",
    "",
    f"- Dataset rows: {len(df_raw):,}",
    f"- Dataset columns: {df_raw.shape[1]:,}",
    f"- Schema matches expected columns: {schema_is_valid}",
    f"- Missing tweet_text values: {int(df_raw['tweet_text'].isna().sum()):,}",
    f"- Blank or missing tweet_text rows: {int(blank_or_missing_text.sum()):,}",
    f"- Exact duplicate rows: {int(df_raw.duplicated().sum()):,}",
    f"- Unique entities: {df_raw['entity'].nunique(dropna=True):,}",
    f"- Unique sentiment labels: {df_raw['sentiment'].nunique(dropna=True):,}",
    f"- Extra raw label outside assignment scope: {', '.join(extra_sentiments) if extra_sentiments else 'None'}",
    "",
    "## Raw Sentiment Distribution",
    "",
]
phase2_summary_lines.extend(
    [
        f"- {row.sentiment}: {int(row.count):,} ({row.percent:.2f}%)"
        for row in sentiment_distribution.itertuples(index=False)
    ]
)
phase2_summary_lines.extend(
    [
        "",
        "## Recommendation For Phase 3",
        "",
        "Create a cleaned working DataFrame by dropping blank or missing tweet text, removing exact duplicate rows, documenting exclusion of Irrelevant, and then applying tweet text cleaning.",
    ]
)
summary_path = REPORTS_DIR / "phase2_validation_summary.md"
summary_path.write_text("\n".join(phase2_summary_lines) + "\n", encoding="utf-8")

print(f"Saved {len(phase2_tables)} Phase 2 tables to {TABLES_DIR}")
print(f"Saved Phase 2 validation summary to {summary_path}")

## Phase 2 Findings To Carry Forward

The raw dataset loads successfully into four columns: `tweet_id`, `entity`, `sentiment`, and `tweet_text`.

Important findings to use in Phase 3:

- The CSV has no header row, so explicit column names are required.
- The raw dataset includes a fourth label, `Irrelevant`, even though the assignment asks for `Negative`, `Neutral`, and `Positive` sentiment classification.
- Blank or missing tweet text and duplicate rows must be handled before modeling.
- Text length statistics should guide sequence padding choices in the RNN phase.

Recommended Phase 3 action: create a cleaned working DataFrame that drops blank or missing tweet text, removes exact duplicate rows, documents the exclusion of `Irrelevant`, and prepares text-cleaning functions for URLs, mentions, hashtags, special characters, tokenization, stop-word removal, and lemmatization or stemming.

# Part 2: Exploratory Data Analysis

Planned work:

- Summary statistics.
- Sentiment distribution.
- Entity/topic distribution.
- Tweet length analysis.
- Top word frequency by sentiment.
- Word clouds for positive and negative tweets.
- Written EDA insights.

# Part 3: RNN Model Development

Planned work:

- Create train, validation, and test splits.
- Build vocabulary from training data only.
- Convert tweets to padded token sequences.
- Train an embedding-based LSTM or GRU model.
- Use dropout and controlled regularization.
- Track learning curves across epochs.

# Part 4: Evaluation, Improvement, And Presentation

Planned work:

- Report accuracy, precision, recall, and F1-score.
- Plot confusion matrix and learning curves.
- Compare baseline and improved RNN models.
- Run a small hyperparameter search.
- Demonstrate predictions on sample tweets.
- Summarize findings in a presentation-ready format.